# Level 2: E-Commerce Sales Analysis 📊📈

This notebook contains the complete Exploratory Data Analysis (EDA) and Business Insights pipeline for the E-Commerce Sales dataset. 
Developed as part of the **Data Science Internship** at **Aadyam Talent Consultancy (ATC)**.

### **Objectives:**
1. **Data Cleaning:** Detect and resolve missing values, duplicate entries, and inconsistent categorical entries.
2. **Sales Performance:** Identify annual revenue, net profits, and AOV.
3. **Seasonality:** Pinpoint sales fluctuations over months and quarters.
4. **Product & Regional Analysis:** Track category distribution and geographical top performers.

In [ ]:
# Imports and Configuration
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set custom plotting styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['grid.alpha'] = 0.3

## 1. Load and Inspect Raw Data

In [ ]:
# Load the generated raw e-commerce transaction dataset
raw_df = pd.read_csv('dataset/ecommerce_sales_raw.csv')
print(f"Raw dataset dimensions: {raw_df.shape}")
print(f"Number of exact duplicates: {raw_df.duplicated().sum()}")
print(f"Missing product names: {raw_df['Product Name'].isna().sum()}")
print(f"Missing postal codes: {raw_df['Postal Code'].isna().sum()}")
raw_df.head(3)

## 2. Execute Data Cleaning Pipeline

In [ ]:
from src.data_cleaner import clean_data

# Run cleaning and engineering pipeline
df = clean_data('dataset/ecommerce_sales_raw.csv', 'dataset/ecommerce_sales_clean.csv')
print(f"\nCleaned dataset shape: {df.shape}")
print(f"Remaining missing Product Names: {df['Product Name'].isna().sum()}")
print(f"Remaining missing Postal Codes: {df['Postal Code'].isna().sum()}")

## 3. High-Level Key Performance Indicators (KPIs)

In [ ]:
# KPI calculations
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
margin = (total_profit / total_sales) * 100
total_orders = df['Order ID'].nunique()
aov = total_sales / total_orders

print("=== KEY PERFORMANCE INDICATORS ===")
print(f"Total Sales Revenue : ${total_sales:,.2f}")
print(f"Total Net Profit    : ${total_profit:,.2f}")
print(f"Profit Margin       : {margin:.2f}%")
print(f"Total Orders        : {total_orders:,}")
print(f"Average Order Value : ${aov:,.2f}")

## 4. Visualizing Analysis Insights

### A. Monthly Sales & Net Profit Trends

In [ ]:
monthly = df.groupby(['Order Month Num', 'Order Month']).agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum')
).reset_index().sort_values('Order Month Num')

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(monthly['Order Month'], monthly['Sales'], color='#1A365D', marker='o', linewidth=2.5, label='Sales')
ax2.plot(monthly['Order Month'], monthly['Profit'], color='#0D9488', marker='s', linewidth=2.0, linestyle='--', label='Profit')

ax1.set_xlabel('Month', fontweight='bold')
ax1.set_ylabel('Sales ($)', color='#1A365D', fontweight='bold')
ax2.set_ylabel('Profit ($)', color='#0D9488', fontweight='bold')
ax1.set_xticklabels(monthly['Order Month'], rotation=30, ha='right')
plt.title('Monthly Sales and Net Profit Trends', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### B. Product Category Revenue Distribution

In [ ]:
category = df.groupby('Category')['Sales'].sum().reset_index()

plt.figure(figsize=(6, 6))
plt.pie(
    category['Sales'],
    labels=category['Category'],
    autopct='%1.1f%%',
    startangle=140,
    colors=['#1A365D', '#0D9488', '#F59E0B'],
    textprops=dict(fontweight='semibold')
)
centre_circle = plt.Circle((0,0), 0.55, fc='white')
plt.gcf().gca().add_artist(centre_circle)
plt.title('Sales Distribution by Product Category', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### C. Geographical Regional Performance

In [ ]:
region = df.groupby('Region').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum')
).reset_index()

region_melted = pd.melt(region, id_vars=['Region'], value_vars=['Sales', 'Profit'], var_name='Metric', value_name='Value')

plt.figure(figsize=(9, 5))
sns.barplot(x='Region', y='Value', hue='Metric', data=region_melted, palette=['#1A365D', '#0D9488'])
plt.title('Sales Revenue vs Net Profit by Geographical Region', fontweight='bold', fontsize=13)
plt.ylabel('Value ($)', fontweight='bold')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## 5. Main Business Insights & Key Takeaways

1. **Technology Dominance:** Technology product lines generate the vast majority of sales revenue (~58.8%), highlighting a clear area to invest marketing and promotion resources.
2. **Q4 Surge Seasonality:** A major retail surge is noticeable in November and December, contributing over 27% of annual sales. Supply chain and inventories should prepare in October.
3. **Profitability Margin:** An overall net profit margin of ~20% indicates strong pricing models, but Technology margins are slightly lower, indicating discount structures on tech items should be audited.